# 00 - Practice: World Bank API -> Postgres

Практика по ingestion-пайплайну: забираем страны из World Bank API, складываем raw JSON в Postgres, затем нормализуем данные в отдельную таблицу.

Что делаем:
- проверяем подключение к Postgres и доступность API;
- создаем raw-таблицу с `JSONB`;
- загружаем данные страницами с идемпотентным upsert;
- нормализуем данные в `countries_dim`;
- запускаем простые DQ-проверки и смотрим результат.

## Run

```bash
cp infra/env/.env.example infra/env/.env
make up STACK=postgres_jupyter
make smoke STACK=postgres_jupyter
```

Jupyter: `http://localhost:8888`
Token: значение `JUPYTER_TOKEN` из `infra/env/.env`.

In [ ]:
import json
import os
import time
from typing import Iterable, List

import pandas as pd
import requests
from sqlalchemy import create_engine, text

pd.set_option("display.max_columns", 50)
pd.set_option("display.max_colwidth", 120)

In [ ]:
POSTGRES_USER = os.getenv("POSTGRES_USER", "course")
POSTGRES_PASSWORD = os.getenv("POSTGRES_PASSWORD", "course")
POSTGRES_DB = os.getenv("POSTGRES_DB", "course")
POSTGRES_HOST = os.getenv("POSTGRES_HOST", "postgres")
POSTGRES_PORT = os.getenv("POSTGRES_PORT", "5432")

WORLD_BANK_API_BASE = os.getenv("WORLD_BANK_API_BASE", "https://api.worldbank.org/v2").rstrip("/")
WORLD_BANK_COUNTRY_URL = f"{WORLD_BANK_API_BASE}/country"

PER_PAGE = 100
SLEEP_SEC = 0.2
REQUEST_TIMEOUT = 30

PG_URL = (
    f"postgresql+psycopg2://{POSTGRES_USER}:{POSTGRES_PASSWORD}"
    f"@{POSTGRES_HOST}:{POSTGRES_PORT}/{POSTGRES_DB}"
)

engine = create_engine(PG_URL)
PG_URL.replace(POSTGRES_PASSWORD, "***")

## Connectivity checks

In [ ]:
with engine.begin() as conn:
    print(conn.execute(text("select version()")).scalar())

In [ ]:
response = requests.get(
    WORLD_BANK_COUNTRY_URL,
    params={"format": "json", "per_page": 1, "page": 1},
    timeout=REQUEST_TIMEOUT,
)
response.raise_for_status()
response.status_code, response.headers.get("content-type", ""), response.json()[0]

## Table setup

In [ ]:
DDL = """
create table if not exists raw_wb_countries (
    id text primary key,
    payload jsonb not null,
    ingested_at timestamptz not null default now()
);

create table if not exists countries_dim (
    id text primary key,
    iso2_code text,
    name text not null,
    region_id text,
    region_name text,
    income_id text,
    income_name text,
    lending_id text,
    lending_name text,
    capital_city text,
    longitude double precision,
    latitude double precision,
    loaded_at timestamptz not null default now()
);
"""

def ensure_tables() -> None:
    with engine.begin() as conn:
        conn.execute(text(DDL))

ensure_tables()
print("Tables are ready")

## Ingest raw data
World Bank API возвращает массив вида `[meta, items]`, где `meta` содержит пагинацию, а `items` - список стран.

In [ ]:
INSERT_RAW = text(
    """
    insert into raw_wb_countries (id, payload)
    values (:id, cast(:payload as jsonb))
    on conflict (id) do update set
        payload = excluded.payload,
        ingested_at = now()
    """
)

def wb_fetch_countries(per_page: int = PER_PAGE, sleep_sec: float = SLEEP_SEC) -> Iterable[List[dict]]:
    page = 1

    while True:
        response = requests.get(
            WORLD_BANK_COUNTRY_URL,
            params={"format": "json", "per_page": per_page, "page": page},
            timeout=REQUEST_TIMEOUT,
        )
        response.raise_for_status()
        payload = response.json()

        if not isinstance(payload, list) or len(payload) < 2:
            raise ValueError(f"Unexpected World Bank response on page {page}: {payload}")

        meta, items = payload
        pages = int(meta.get("pages", 1))
        yield items

        if page >= pages:
            break

        page += 1
        time.sleep(sleep_sec)

def load_raw(per_page: int = PER_PAGE) -> int:
    loaded = 0

    with engine.begin() as conn:
        for batch in wb_fetch_countries(per_page=per_page):
            for item in batch:
                country_id = item.get("id")
                if not country_id:
                    continue

                conn.execute(
                    INSERT_RAW,
                    {"id": country_id, "payload": json.dumps(item, ensure_ascii=False)},
                )
                loaded += 1

    return loaded

In [ ]:
loaded_raw = load_raw()
print(f"Raw rows processed: {loaded_raw}")

## Normalize into `countries_dim`

In [ ]:
UPSERT_DIM = text(
    """
    insert into countries_dim (
        id, iso2_code, name,
        region_id, region_name,
        income_id, income_name,
        lending_id, lending_name,
        capital_city, longitude, latitude
    )
    values (
        :id, :iso2_code, :name,
        :region_id, :region_name,
        :income_id, :income_name,
        :lending_id, :lending_name,
        :capital_city, :longitude, :latitude
    )
    on conflict (id) do update set
        iso2_code = excluded.iso2_code,
        name = excluded.name,
        region_id = excluded.region_id,
        region_name = excluded.region_name,
        income_id = excluded.income_id,
        income_name = excluded.income_name,
        lending_id = excluded.lending_id,
        lending_name = excluded.lending_name,
        capital_city = excluded.capital_city,
        longitude = excluded.longitude,
        latitude = excluded.latitude,
        loaded_at = now()
    """
)

SELECT_RAW = text("select payload from raw_wb_countries order by id")

def as_float(value):
    try:
        return float(value) if value not in (None, "") else None
    except (TypeError, ValueError):
        return None

def load_dim() -> int:
    loaded = 0

    with engine.begin() as conn:
        for payload in conn.execute(SELECT_RAW).scalars():
            item = payload if isinstance(payload, dict) else json.loads(payload)
            conn.execute(
                UPSERT_DIM,
                {
                    "id": item.get("id"),
                    "iso2_code": item.get("iso2Code"),
                    "name": item.get("name") or item.get("value") or "(unknown)",
                    "region_id": (item.get("region") or {}).get("id"),
                    "region_name": (item.get("region") or {}).get("value"),
                    "income_id": (item.get("incomeLevel") or {}).get("id"),
                    "income_name": (item.get("incomeLevel") or {}).get("value"),
                    "lending_id": (item.get("lendingType") or {}).get("id"),
                    "lending_name": (item.get("lendingType") or {}).get("value"),
                    "capital_city": item.get("capitalCity"),
                    "longitude": as_float(item.get("longitude")),
                    "latitude": as_float(item.get("latitude")),
                },
            )
            loaded += 1

    return loaded

In [ ]:
loaded_dim = load_dim()
print(f"Dimension rows processed: {loaded_dim}")

## DQ checks

In [ ]:
DQ_QUERY = text(
    """
    select
        (select count(*) from raw_wb_countries) as raw_cnt,
        (select count(*) from countries_dim) as dim_cnt,
        (select count(*) from countries_dim where name is null) as null_name_cnt,
        (select count(*) from countries_dim where iso2_code is null) as null_iso2_code_cnt,
        (select count(*) - count(distinct id) from countries_dim) as dup_ids
    """
)

def run_dq() -> pd.DataFrame:
    with engine.begin() as conn:
        row = conn.execute(DQ_QUERY).mappings().one()
    return pd.DataFrame([row])

run_dq()

## Inspect results

In [ ]:
pd.read_sql(
    text("""
    select id, ingested_at, left(payload::text, 120) as payload_preview
    from raw_wb_countries
    order by ingested_at desc, id
    limit 10
    """),
    engine,
)

In [ ]:
pd.read_sql(
    text("""
    select id, iso2_code, name, region_name, income_name, capital_city
    from countries_dim
    order by name
    limit 20
    """),
    engine,
)

In [ ]:
pd.read_sql(
    text("""
    select region_name, count(*) as countries_cnt
    from countries_dim
    group by region_name
    order by countries_cnt desc, region_name
    """),
    engine,
)